# Spatial MRV Maps And Reports

This notebook turns the processed ERW/SCEPTER outputs into map-ready and report-ready products. It starts after `06_extract_scepter_outputs.ipynb`, using spatial model-unit results joined back to cropland parcels.

The purpose is to produce clear MRV deliverables: scenario comparison tables, spatial maps, AOI-level summaries, and a short Markdown report that records what was mapped and which outputs are still missing.


## Workflow

1. **Load spatial SCEPTER results** from `data/scepter_runs/data/outputs/extracted/scepter_results_by_model_unit.gpkg`.
2. **Load summary tables** from notebook `06` and cropland validation where available.
3. **Choose MRV metrics** such as cumulative CO2 removal or CO2 removal per hectare when present.
4. **Build scenario comparison tables** for baseline and ERW scenarios.
5. **Create map layers** for each scenario and each available MRV metric.
6. **Write map images and vector layers** for reports and GIS review.
7. **Write a Markdown report** summarizing inputs, status, metrics, outputs, and next decisions.

If SCEPTER has not been executed yet, this notebook still produces a status report showing that result summaries are missing. Once SCEPTER outputs exist, rerun the notebook to produce quantitative MRV maps.


In [ ]:
from pathlib import Path
import os
import sys

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

def mount_google_drive_if_colab() -> None:
    try:
        from google.colab import drive
    except ModuleNotFoundError:
        return

    drive.mount("/content/drive")


mount_google_drive_if_colab()

COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/erw_spatial_mrv")
COLAB_DATA_ROOT = COLAB_PROJECT_ROOT / "data"
LOCAL_PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()


def has_erw_package(project_root: Path) -> bool:
    return (project_root / "src" / "erw_mrv" / "__init__.py").exists()


def source_root_candidates() -> list[Path]:
    cwd = Path.cwd().resolve()
    candidates = [LOCAL_PROJECT_ROOT, COLAB_PROJECT_ROOT]
    for base in (cwd, *cwd.parents):
        candidates.extend((base, base / "erw_spatial_mrv"))
    candidates.extend(
        Path(path)
        for path in (
            "/content/erw_spatial_mrv",
            "/content/enhanced_rock_weathering/erw_spatial_mrv",
            "/content/drive/MyDrive/erw_spatial_mrv",
        )
    )
    unique = []
    for candidate in candidates:
        if candidate not in unique:
            unique.append(candidate)
    return unique


def find_source_project_root() -> Path:
    for candidate in source_root_candidates():
        if has_erw_package(candidate):
            return candidate
    checked = chr(10).join(f"- {candidate}" for candidate in source_root_candidates())
    raise ModuleNotFoundError(
        "Could not find src/erw_mrv. The data can live in Google Drive, but "
        "the notebook still needs the project source folder containing src/erw_mrv. "
        "In Colab, upload/sync the full erw_spatial_mrv project or run from a "
        "checkout that includes src/. Checked: "
        f"{checked}"
    )


SOURCE_PROJECT_ROOT = find_source_project_root()
PROJECT_ROOT = COLAB_PROJECT_ROOT if COLAB_DATA_ROOT.exists() else SOURCE_PROJECT_ROOT
DATA_ROOT = COLAB_DATA_ROOT if COLAB_DATA_ROOT.exists() else PROJECT_ROOT / "data"
os.environ["ERW_MRV_DATA_ROOT"] = str(DATA_ROOT)

SRC = SOURCE_PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"SOURCE_PROJECT_ROOT = {SOURCE_PROJECT_ROOT}")
print(f"DATA_ROOT = {DATA_ROOT}")

from erw_mrv.paths import (
    OUTPUT_FIGURES,
    OUTPUT_MAPS,
    OUTPUT_REPORTS,
    OUTPUT_TABLES,
    SCEPTER_OUTPUTS,
    ensure_dir,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)
plt.rcParams["figure.figsize"] = (10, 8)


## Configure Inputs And Outputs


In [ ]:
EXTRACTED_DIR = SCEPTER_OUTPUTS / "extracted"
SPATIAL_RESULTS_PATH = EXTRACTED_DIR / "scepter_results_by_model_unit.gpkg"
RESULTS_TABLE_PATH = EXTRACTED_DIR / "scepter_results_long.csv"
SCENARIO_SUMMARY_PATH = EXTRACTED_DIR / "scepter_scenario_summary.csv"
ADDITIONALITY_PATH = EXTRACTED_DIR / "scepter_additionality.csv"
CROPLAND_SUMMARY_PATH = PROJECT_ROOT / "data" / "processed" / "landcover" / "validation" / "jan_jun_2026" / "cropland_summary_jan_jun_2026.csv"

MAP_DIR = ensure_dir(OUTPUT_MAPS / "spatial_mrv")
FIGURE_DIR = ensure_dir(OUTPUT_FIGURES / "spatial_mrv")
TABLE_DIR = ensure_dir(OUTPUT_TABLES / "spatial_mrv")
REPORT_DIR = ensure_dir(OUTPUT_REPORTS / "spatial_mrv")

required = [SPATIAL_RESULTS_PATH, RESULTS_TABLE_PATH]
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing notebook 06 output(s). Run notebook 06 first. Missing: "
        + "; ".join(str(path) for path in missing)
    )

MAP_DIR, FIGURE_DIR, TABLE_DIR, REPORT_DIR


## Load Results


In [ ]:
spatial_results = gpd.read_file(SPATIAL_RESULTS_PATH)
results = pd.read_csv(RESULTS_TABLE_PATH)
scenario_summary = pd.read_csv(SCENARIO_SUMMARY_PATH) if SCENARIO_SUMMARY_PATH.exists() else pd.DataFrame()
additionality = pd.read_csv(ADDITIONALITY_PATH) if ADDITIONALITY_PATH.exists() else pd.DataFrame()
cropland_summary = pd.read_csv(CROPLAND_SUMMARY_PATH) if CROPLAND_SUMMARY_PATH.exists() else pd.DataFrame()

print(f"Spatial result rows: {len(spatial_results):,}")
print(f"Flat result rows: {len(results):,}")
print(f"Scenarios: {spatial_results['scenario_id'].nunique() if 'scenario_id' in spatial_results.columns else 0:,}")
spatial_results.head()


## Select MRV Metrics

The metric names will depend on the final SCEPTER output parser. This cell searches for common CO2 removal and weathering/alkalinity columns and uses whatever is available.


In [ ]:
PREFERRED_MAP_METRICS = [
    "additional_co2_t",
    "co2_removal_t_ha",
    "co2_removal_t",
    "cumulative_co2_removal_t",
    "net_co2_removal_t",
    "alkalinity_mol_ha",
    "weathering_rate",
]

available_metrics = [
    column for column in PREFERRED_MAP_METRICS
    if column in spatial_results.columns and pd.api.types.is_numeric_dtype(spatial_results[column])
]

if not available_metrics:
    numeric_candidates = [
        column for column in spatial_results.select_dtypes(include="number").columns
        if column not in {"area_ha", "centroid_lon", "centroid_lat"}
    ]
    available_metrics = numeric_candidates[:3]

PRIMARY_METRIC = available_metrics[0] if available_metrics else None
print("Available MRV map metrics:", available_metrics)
print("Primary metric:", PRIMARY_METRIC)


## Status And Scenario Tables

These tables are useful even before quantitative SCEPTER outputs exist, because they show whether model runs were staged, completed, or missing summary files.


In [ ]:
status_summary = results.groupby(["scenario_id", "execution_status", "result_status"]).size().rename("run_count").reset_index()
status_summary_path = TABLE_DIR / "scepter_result_status_summary.csv"
status_summary.to_csv(status_summary_path, index=False)

if PRIMARY_METRIC:
    scenario_metric_summary = spatial_results.groupby("scenario_id").agg(
        model_units=("model_unit_id", "nunique"),
        total_area_ha=("area_ha", "sum"),
        metric_mean=(PRIMARY_METRIC, "mean"),
        metric_sum=(PRIMARY_METRIC, "sum"),
    ).reset_index()
    scenario_metric_summary["metric"] = PRIMARY_METRIC
else:
    scenario_metric_summary = spatial_results.groupby("scenario_id").agg(
        model_units=("model_unit_id", "nunique"),
        total_area_ha=("area_ha", "sum"),
    ).reset_index()
    scenario_metric_summary["metric"] = "none_available"

scenario_metric_path = TABLE_DIR / "spatial_mrv_scenario_summary.csv"
scenario_metric_summary.to_csv(scenario_metric_path, index=False)

status_summary, scenario_metric_summary


## Export Spatial Result Layer

This GeoPackage is the compact GIS handoff from the MRV workflow.


In [ ]:
spatial_export_path = MAP_DIR / "spatial_mrv_scepter_results.gpkg"
spatial_results.to_file(spatial_export_path, layer="scepter_results", driver="GPKG")
spatial_export_path


## Create Scenario Maps

Maps are exported as PNGs. If no numeric MRV metric exists yet, the notebook maps run status instead.


In [ ]:
map_paths = []
scenario_ids = sorted(spatial_results["scenario_id"].dropna().unique()) if "scenario_id" in spatial_results.columns else ["all"]

for scenario_id in scenario_ids:
    scenario_gdf = spatial_results[spatial_results["scenario_id"] == scenario_id].copy()
    if scenario_gdf.empty:
        continue

    fig, ax = plt.subplots(figsize=(10, 8))
    if PRIMARY_METRIC:
        scenario_gdf.plot(
            column=PRIMARY_METRIC,
            ax=ax,
            legend=True,
            cmap="viridis",
            linewidth=0.1,
            edgecolor="black",
        )
        title = f"{scenario_id}: {PRIMARY_METRIC}"
    elif "result_status" in scenario_gdf.columns:
        scenario_gdf.plot(
            column="result_status",
            ax=ax,
            legend=True,
            categorical=True,
            linewidth=0.1,
            edgecolor="black",
        )
        title = f"{scenario_id}: result status"
    else:
        scenario_gdf.plot(ax=ax, color="#8ecae6", edgecolor="#023047", linewidth=0.1)
        title = f"{scenario_id}: model units"

    ax.set_title(title)
    ax.set_axis_off()
    out_path = MAP_DIR / f"spatial_mrv_{scenario_id}.png"
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    map_paths.append(out_path)

map_paths


## Scenario Comparison Figure

This chart compares scenarios using the primary metric when available. Otherwise it shows available/staged run counts.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
if PRIMARY_METRIC:
    plot_data = scenario_metric_summary.sort_values("metric_sum", ascending=False)
    ax.bar(plot_data["scenario_id"], plot_data["metric_sum"])
    ax.set_ylabel(f"Sum of {PRIMARY_METRIC}")
    ax.set_title("Scenario comparison")
else:
    plot_data = status_summary.groupby("scenario_id")["run_count"].sum().reset_index()
    ax.bar(plot_data["scenario_id"], plot_data["run_count"])
    ax.set_ylabel("Run count")
    ax.set_title("Scenario run counts")

ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
comparison_path = FIGURE_DIR / "spatial_mrv_scenario_comparison.png"
fig.savefig(comparison_path, dpi=200, bbox_inches="tight")
plt.show()
comparison_path


## District/AOI MRV Summary

This table is intentionally compact: it is the kind of summary that can go into a methods note or MRV report appendix.


In [ ]:
report_summary = {
    "spatial_result_rows": int(len(spatial_results)),
    "scenario_count": int(len(scenario_ids)),
    "primary_metric": PRIMARY_METRIC or "none_available",
    "parsed_runs": int((results["result_status"] == "parsed").sum()) if "result_status" in results.columns else 0,
    "missing_summaries": int((results["result_status"] == "missing_summary").sum()) if "result_status" in results.columns else 0,
}

if not cropland_summary.empty and {"metric", "value"}.issubset(cropland_summary.columns):
    for _, row in cropland_summary.iterrows():
        key = str(row["metric"]).lower().replace(" ", "_")
        report_summary[f"cropland_{key}"] = row["value"]

report_summary_df = pd.DataFrame([report_summary])
report_summary_path = TABLE_DIR / "spatial_mrv_report_summary.csv"
report_summary_df.to_csv(report_summary_path, index=False)
report_summary_df


## Write Markdown Report

The report is deliberately short and operational: it records what exists, what was mapped, and what remains blocked.


In [ ]:
report_path = REPORT_DIR / "spatial_mrv_report.md"
map_list = "\n".join(f"- `{path.relative_to(PROJECT_ROOT)}`" for path in map_paths) or "- No maps generated."
metric_text = PRIMARY_METRIC or "No numeric MRV metric available yet."

report = f"""# Spatial MRV Report

## Inputs

- Spatial SCEPTER results: `{SPATIAL_RESULTS_PATH.relative_to(PROJECT_ROOT)}`
- Flat SCEPTER results: `{RESULTS_TABLE_PATH.relative_to(PROJECT_ROOT)}`
- Scenario summary: `{SCENARIO_SUMMARY_PATH.relative_to(PROJECT_ROOT) if SCENARIO_SUMMARY_PATH.exists() else 'not available'}`
- Cropland summary: `{CROPLAND_SUMMARY_PATH.relative_to(PROJECT_ROOT) if CROPLAND_SUMMARY_PATH.exists() else 'not available'}`

## Result Status

- Spatial result rows: {report_summary['spatial_result_rows']:,}
- Scenario count: {report_summary['scenario_count']:,}
- Parsed SCEPTER runs: {report_summary['parsed_runs']:,}
- Missing SCEPTER summaries: {report_summary['missing_summaries']:,}
- Primary mapped metric: `{metric_text}`

## Outputs

- Spatial result layer: `{spatial_export_path.relative_to(PROJECT_ROOT)}`
- Scenario summary table: `{scenario_metric_path.relative_to(PROJECT_ROOT)}`
- Status summary table: `{status_summary_path.relative_to(PROJECT_ROOT)}`
- Report summary table: `{report_summary_path.relative_to(PROJECT_ROOT)}`
- Scenario comparison figure: `{comparison_path.relative_to(PROJECT_ROOT)}`

## Maps

{map_list}

## Notes

If parsed SCEPTER runs are zero, the pipeline is still in staged mode. Run notebook `05` with a valid external SCEPTER command, then rerun notebook `06` and this notebook to produce quantitative CO2 removal maps.
"""

report_path.write_text(report, encoding="utf-8")
print(report_path)
print(report[:2000])


## Outputs From This Notebook

This notebook writes spatial MRV products under `data/outputs/`:

- `data/outputs/maps/spatial_mrv/spatial_mrv_scepter_results.gpkg`: GIS-ready model-unit/scenario result layer.
- `data/outputs/maps/spatial_mrv/spatial_mrv_{scenario_id}.png`: one map per scenario.
- `data/outputs/figures/spatial_mrv/spatial_mrv_scenario_comparison.png`: scenario comparison chart.
- `data/outputs/tables/spatial_mrv/scepter_result_status_summary.csv`: run/result status counts.
- `data/outputs/tables/spatial_mrv/spatial_mrv_scenario_summary.csv`: scenario-level MRV metric summary.
- `data/outputs/tables/spatial_mrv/spatial_mrv_report_summary.csv`: compact report header metrics.
- `data/outputs/reports/spatial_mrv/spatial_mrv_report.md`: Markdown report for review.

The next step is uncertainty/sensitivity analysis: vary application rate, particle size, runoff, soil chemistry, and model parameters to produce uncertainty intervals around the spatial MRV estimates.
